# Visão Computacional

**Tarefa:** _[Classificação / Detecção / Segmentação]_  
**Dataset:** _[nome / fonte]_  
**Objetivo:** _[o que o modelo deve identificar nas imagens?]_

## 1. Imports e Dispositivo

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
torch.manual_seed(SEED)
print(f'Dispositivo: {DEVICE}')

## 2. Transformações e Dataset

In [ ]:
# Transformações padrão para modelos pré-treinados (ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_treino = T.Compose([
    T.Resize((256, 256)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_teste = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Carregamento:
# dataset_treino = datasets.ImageFolder('../data/imagens/treino/', transform=transform_treino)
# dataset_teste  = datasets.ImageFolder('../data/imagens/teste/',  transform=transform_teste)

# Exemplo com CIFAR-10 para testar sem dados próprios:
dataset_treino = datasets.CIFAR10(root='../data/', train=True,  download=True, transform=transform_treino)
dataset_teste  = datasets.CIFAR10(root='../data/', train=False, download=True, transform=transform_teste)

loader_treino = DataLoader(dataset_treino, batch_size=32, shuffle=True,  num_workers=0)
loader_teste  = DataLoader(dataset_teste,  batch_size=32, shuffle=False, num_workers=0)

CLASSES     = dataset_treino.classes
NUM_CLASSES = len(CLASSES)
print(f'Classes ({NUM_CLASSES}): {CLASSES}')

## 3. Visualizar Amostras

In [ ]:
def desnormalizar(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

imgs, labels = next(iter(loader_treino))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = desnormalizar(imgs[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASSES[labels[i]])
    ax.axis('off')
plt.suptitle('Amostras do Dataset')
plt.tight_layout()
plt.show()

## 4. Modelo com Transfer Learning

In [ ]:
# EfficientNet-B0 pré-treinado (leve, bom para CPU)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Congelar backbone — treinar só o classificador final
for param in model.features.parameters():
    param.requires_grad = False

# Substituir cabeça de classificação
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
treinaveis       = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total de parâmetros: {total_params:,}')
print(f'Treináveis:          {treinaveis:,}')

## 5. Configuração do Treino

In [ ]:
EPOCHS    = 10
LR        = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 6. Loop de Treino

In [ ]:
def avaliar(loader):
    model.eval()
    corretos, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            corretos += (preds == labels).sum().item()
            total    += labels.size(0)
    return corretos / total


hist = []
for epoca in range(1, EPOCHS + 1):
    model.train()
    loss_total = 0.0
    for imgs, labels in loader_treino:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item()
    scheduler.step()
    acc_val = avaliar(loader_teste)
    hist.append({'epoca': epoca, 'loss': loss_total/len(loader_treino), 'acc_val': acc_val})
    print(f'Época {epoca:2d} | Loss: {loss_total/len(loader_treino):.4f} | Acc val: {acc_val:.4f}')

## 7. Curvas de Aprendizado

In [ ]:
import pandas as pd
hist_df = pd.DataFrame(hist)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
hist_df.plot(x='epoca', y='loss',    ax=ax1, legend=False)
ax1.set_title('Loss de Treino')
hist_df.plot(x='epoca', y='acc_val', ax=ax2, legend=False, color='orange')
ax2.set_title('Acurácia de Validação')
plt.tight_layout()
plt.show()

## 8. Conclusões

- _Performance final no conjunto de teste_
- _Classes com mais erro_
- _Próximos passos (fine-tuning completo, data augmentation, modelos maiores)_